In [117]:
! pip install feature_engine==1.8.0 --quiet

In [118]:
!pip install optuna==4.8.0 -q

In [349]:
!pip uninstall scikit-learn -y
!pip install scikit-learn==1.5.2

Found existing installation: scikit-learn 1.5.2
Uninstalling scikit-learn-1.5.2:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/scikit_learn-1.5.2.dist-info/*
    /usr/local/lib/python3.12/dist-packages/scikit_learn.libs/libgomp-a34b3233.so.1.0.0
    /usr/local/lib/python3.12/dist-packages/sklearn/*
Proceed (Y/n)? ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/uninstall.py", line 106, in run
    uninstall_pathset = req.uninstall(
                        ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/req/req_install.py", line 722, in uninstall
    uninstalled_pathset.remove(auto_confirm, verbose)
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/req/req_uninstall.py", 

In [350]:
import pandas as pd
import numpy as np
import sys
import os
import gc

import unicodedata
import re

import seaborn as sns
import matplotlib.pyplot as plt

import feature_engine
from feature_engine.selection import (
    RecursiveFeatureElimination,
    DropConstantFeatures,
    DropDuplicateFeatures,
)
import optuna

from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, make_scorer
from sklearn.preprocessing import StandardScaler
import sklearn

import xgboost
import lightgbm
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import time

print(f"Versão:")
print(f'Python: {sys.version}')
print(f'Pandas: {pd.__version__}')
print(f'Numpy: {np.__version__}')
print("Feature Engine:", feature_engine.__version__)
print("Scikit Learn:", sklearn.__version__)
print("Optuna:", optuna.__version__)
print("XGBoost:", xgboost.__version__)
print("LightGBM:", lightgbm.__version__)

Versão:
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Pandas: 2.2.2
Numpy: 2.0.2
Feature Engine: 1.8.0
Scikit Learn: 1.6.1
Optuna: 4.8.0
XGBoost: 3.2.0
LightGBM: 4.6.0


In [175]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
path_main = '/content/drive/MyDrive/ML-UFF'

Mounted at /content/drive


In [176]:
def get_data_frame(path_dataset: str):
  data = pd.read_csv(path_dataset)
  return data

In [233]:
path = f'{path_main}/df_agg.csv'
df = get_data_frame(path)

In [234]:
df.rename(columns={'classificacao': 'total_casos','Data':'data'}, inplace=True)

In [235]:
df['data'] = pd.to_datetime(df['data'])
df.set_index('data', inplace=True)

In [236]:
df.shape, df.columns

((92794, 44),
 Index(['cs_sexo', 'cs_gestant', 'febre', 'mialgia', 'cefaleia', 'exantema',
        'vomito', 'nausea', 'dor_retro', 'artralgia', 'artrite', 'conjuntvit',
        'petequia_n', 'laco', 'diabetes', 'hipertensa', 'renal', 'hematolog',
        'hepatopat', 'classi_fin', 'evolucao', 'idade', 'faixa_etaria', 'year',
        'month', 'day_week', 'day_month', 'day_month_start', 'day_month_end',
        'day_quarter_start', 'day_quarter_end', 'day_year_start',
        'day_year_end', 'total_casos', 'desfecho', 'chuva_dia',
        'temp_media_dia', 'temp_max_dia', 'temp_min_dia', 'umidade_dia',
        'vento_velocidade_horaria_m_s', 'score_gravidade', 'calor_extremo',
        'faixa_umidade'],
       dtype='object'))

In [237]:
def print_nan(df):
    nan_count = df.isna().sum()
    # Filtra apenas onde a contagem é maior que zero
    nan_only = nan_count[nan_count > 0].sort_values(ascending=False)

    if nan_only.empty:
        print("Nenhum valor NaN encontrado!")
    else:
        print("Colunas com valores NaN:")
        print(nan_only)

In [238]:
print_nan(df)

Colunas com valores NaN:
chuva_dia                       177
temp_media_dia                  177
temp_max_dia                    177
temp_min_dia                    177
umidade_dia                     177
vento_velocidade_horaria_m_s    177
faixa_umidade                   177
dtype: int64


In [239]:
df.index

DatetimeIndex(['2020-01-01', '2020-01-01', '2020-01-01', '2020-01-01',
               '2020-01-01', '2020-01-01', '2020-01-01', '2020-01-01',
               '2020-01-01', '2020-01-01',
               ...
               '2026-04-21', '2026-04-21', '2026-04-21', '2026-04-21',
               '2026-04-22', '2026-04-22', '2026-04-22', '2026-04-22',
               '2026-04-22', '2026-04-23'],
              dtype='datetime64[ns]', name='data', length=92794, freq=None)

### Train Test Split

In [240]:
def train_test_split_by_date(
    data: pd.DataFrame | pd.Series,
    start_date: str,
    split_date: str,
    end_date: str | None = None,
):
    if not isinstance(data.index, pd.DatetimeIndex):
        raise ValueError("data must have a DatetimeIndex")

    # garantir ordem temporal
    data = data.sort_index()

    start_date = pd.to_datetime(start_date)
    split_date = pd.to_datetime(split_date)
    end_date = pd.to_datetime(end_date) if end_date else data.index.max()

    if not (start_date < split_date < end_date):
        raise ValueError("start_date < split_date < end_date is required")

    # 🔑 slicing robusto (não depende da data existir no índice)
    start_pos = data.index.searchsorted(start_date, side="left")
    end_pos = data.index.searchsorted(end_date, side="right")

    data = data.iloc[start_pos:end_pos]

    split_pos = data.index.searchsorted(split_date, side="right")
    train = data.iloc[:split_pos]
    test = data.iloc[split_pos:]

    return train, test

In [241]:
lags = [7, 15, 21, 30, 45, 50, 60]
n_nans = lags[-1]
n_nans

60

In [242]:

start_date = df.index.min().strftime('%Y-%m-%d')
split_date_test = '2025-12-24' # 60 dias maior lag gerado
split_date_validation = '2025-08-26' # 60 dias
end_date = df.index.max().strftime('%Y-%m-%d')

start_date, split_date_validation, split_date_test, end_date

('2020-01-01', '2025-08-26', '2025-12-24', '2026-04-23')

In [243]:
df_train, df_test = train_test_split_by_date(df, start_date, split_date_test)

In [244]:
len(df_test.index.unique())

119

In [245]:
df_test.index.unique()

DatetimeIndex(['2025-12-25', '2025-12-26', '2025-12-27', '2025-12-28',
               '2025-12-29', '2025-12-30', '2025-12-31', '2026-01-01',
               '2026-01-02', '2026-01-03',
               ...
               '2026-04-14', '2026-04-15', '2026-04-16', '2026-04-17',
               '2026-04-18', '2026-04-19', '2026-04-20', '2026-04-21',
               '2026-04-22', '2026-04-23'],
              dtype='datetime64[ns]', name='data', length=119, freq=None)

In [246]:
df_train.index, df_test.index

(DatetimeIndex(['2020-01-01', '2020-01-01', '2020-01-01', '2020-01-01',
                '2020-01-01', '2020-01-01', '2020-01-01', '2020-01-01',
                '2020-01-01', '2020-01-01',
                ...
                '2025-12-23', '2025-12-24', '2025-12-24', '2025-12-24',
                '2025-12-24', '2025-12-24', '2025-12-24', '2025-12-24',
                '2025-12-24', '2025-12-24'],
               dtype='datetime64[ns]', name='data', length=91528, freq=None),
 DatetimeIndex(['2025-12-25', '2025-12-25', '2025-12-25', '2025-12-25',
                '2025-12-25', '2025-12-26', '2025-12-26', '2025-12-26',
                '2025-12-26', '2025-12-26',
                ...
                '2026-04-21', '2026-04-21', '2026-04-21', '2026-04-21',
                '2026-04-22', '2026-04-22', '2026-04-22', '2026-04-22',
                '2026-04-22', '2026-04-23'],
               dtype='datetime64[ns]', name='data', length=1266, freq=None))

In [247]:
df_train, df_val = train_test_split_by_date(df_train, start_date, split_date_validation)

In [248]:
len(df_val.index.unique())

120

In [249]:
df_train.index, df_val.index

(DatetimeIndex(['2020-01-01', '2020-01-01', '2020-01-01', '2020-01-01',
                '2020-01-01', '2020-01-01', '2020-01-01', '2020-01-01',
                '2020-01-01', '2020-01-01',
                ...
                '2025-08-25', '2025-08-25', '2025-08-25', '2025-08-25',
                '2025-08-25', '2025-08-25', '2025-08-26', '2025-08-26',
                '2025-08-26', '2025-08-26'],
               dtype='datetime64[ns]', name='data', length=90644, freq=None),
 DatetimeIndex(['2025-08-27', '2025-08-27', '2025-08-27', '2025-08-27',
                '2025-08-27', '2025-08-27', '2025-08-27', '2025-08-27',
                '2025-08-27', '2025-08-27',
                ...
                '2025-12-23', '2025-12-24', '2025-12-24', '2025-12-24',
                '2025-12-24', '2025-12-24', '2025-12-24', '2025-12-24',
                '2025-12-24', '2025-12-24'],
               dtype='datetime64[ns]', name='data', length=884, freq=None))

# Features Train/Validation/Test Split

#### Recriando para que não haja vazamento de dados

In [250]:
column_target = 'total_casos'

In [251]:
def fourier_features(df, columns, periods, n_terms_list):
    t = np.arange(len(df))

    for col in columns:
        for period, n_terms in zip(periods, n_terms_list):
            period = int(period)  # ← garante que não vira array/lista

            for k in range(1, n_terms + 1):
                df[f"{col}_fourier_cos_p{period}_k{k}"] = np.cos(2 * np.pi * k * t / period)
                df[f"{col}_fourier_sin_p{period}_k{k}"] = np.sin(2 * np.pi * k * t / period)

    return df


seasonal_periods = [7, 30.44, 365.25]  # Semanal, Mensal e Anual
n_terms_list = [3, 1, 2] # Complexidade de cada onda


df_train = fourier_features(df_train, ['ciclos'], seasonal_periods, n_terms_list)
df_val = fourier_features(df_val, ['ciclos'], seasonal_periods, n_terms_list)
df_test = fourier_features(df_test, ['ciclos'], seasonal_periods, n_terms_list)


print_nan(df_train)
print_nan(df_val)
print_nan(df_test)


print(df_train.shape)
print(df_val.shape)
df_test.shape

Nenhum valor NaN encontrado!
Nenhum valor NaN encontrado!
Colunas com valores NaN:
chuva_dia                       177
temp_media_dia                  177
temp_max_dia                    177
temp_min_dia                    177
umidade_dia                     177
vento_velocidade_horaria_m_s    177
faixa_umidade                   177
dtype: int64
(90644, 56)
(884, 56)


(1266, 56)

In [252]:
# Lista de sintomas/sinais para o score (ajuste se necessário)
def sinais_gravidade(df):
  sinais_gravidade = [
      'petequia_n', 'laco', 'renal', 'hematolog', 'hepatopat', 'vomito'
  ]

  # Criando a coluna: ela soma quantos desses sinais cada paciente apresenta
  # Se o paciente tem 1 em todos, o score é 6. Se não tem nenhum, é 0.
  df['score_gravidade'] = df[sinais_gravidade].sum(axis=1)
  return df

In [253]:
df_train = sinais_gravidade(df_train)
df_val = sinais_gravidade(df_val)
df_test = sinais_gravidade(df_test)

In [254]:
print(df_train.shape)
print(df_val.shape)
df_test.shape

(90644, 56)
(884, 56)


(1266, 56)

In [255]:
# Agregação Diária: Transformando pacientes em contagens e médias
def agg_diaria(X):
  X = X.groupby('data').agg({
      'total_casos': 'count',
      'score_gravidade': 'mean',     # Gravidade média dos casos do dia
      'idade': 'mean',               # Idade média dos infectados
      'chuva_dia': 'first',          # Já está agregado, pegamos o valor do dia
      'temp_max_dia': 'first',
      'temp_min_dia': 'first',
      'umidade_dia': 'first',
      'year': 'first',
      'month': 'first',
      'day_week': 'first'
  })


  X['score_gravidade'] = X['score_gravidade'].ffill().bfill()
  X['idade'] = X['idade'].ffill().bfill()

  # Para o clima, preenchemos pequenas falhas com interpolação
  X[['chuva_dia', 'temp_min_dia', 'temp_max_dia', 'umidade_dia']] = X[['chuva_dia', 'temp_min_dia', 'temp_max_dia', 'umidade_dia']].interpolate()
  return X

In [256]:
df_train = agg_diaria(df_train)
df_val = agg_diaria(df_val)
df_test = agg_diaria(df_test)

In [257]:
print(df_train.shape)
print(df_val.shape)
df_test.shape

(1893, 10)
(120, 10)


(119, 10)

In [258]:
df_train[column_target]

,total_casos
data,
2020-01-01,11
2020-01-02,18
2020-01-03,19
2020-01-04,5
2020-01-05,9
...,...
2025-08-22,9
2025-08-23,12
2025-08-24,9


In [259]:
def acum_lags(X, lags = [7, 15, 21, 30, 45, 50, 60]):
  for lag in lags:
      # Chuva acumulada nos últimos X dias (Soma móvel)
      X[f'chuva_acum_{lag}d'] = X['chuva_dia'].rolling(window=lag).sum()
      X['faixa_chuva'] = pd.qcut(X[f'chuva_acum_{lag}d'], q=4, labels=['Baixa', 'Média', 'Alta', 'Muito Alta'])

      # Temperatura média defasada (O que aconteceu há X dias)
      X[f'temp_max_lag_{lag}d'] = X['temp_max_dia'].shift(lag)
      X[f'temp_min_lag_{lag}d'] = X['temp_min_dia'].shift(lag)

      # Gravidade defasada (Para ver se a gravidade sobe antes dos casos)
      X[f'gravidade_lag_{lag}d'] = X['score_gravidade'].shift(lag)

      # Cálculo de Médias Móveis (Suavização da curva de casos)
      X[f'casos_mm{lag}d'] = X['total_casos'].rolling(window=lag).mean()

  return X

In [260]:
df_train = acum_lags(df_train, lags)
df_val = acum_lags(df_val, lags)
df_test = acum_lags(df_test, lags)

In [261]:
print(df_train.shape)
print(df_val.shape)
df_test.shape

(1893, 46)
(120, 46)


(119, 46)

In [262]:
def index_transform(df):
  # Transformando a coluna de dados:
  df['year'] = df.index.year
  df['month'] = df.index.month
  df['day_week'] = df.index.day_of_week
  df['day_month'] = df.index.day
  df['day_month_start'] = df.index.is_month_start.astype(int)
  df['day_month_end'] = df.index.is_month_end.astype(int)
  df['day_quarter_start'] = df.index.is_quarter_start.astype(int)
  df['day_quarter_end'] = df.index.is_quarter_end.astype(int)
  df['day_year_start'] = df.index.is_year_start.astype(int)
  df['day_year_end'] = df.index.is_year_end.astype(int)

  return df

In [263]:
df_train = index_transform(df_train)
df_val = index_transform(df_val)
df_test = index_transform(df_test)

In [264]:
print(df_train.shape)
print(df_val.shape)
df_test.shape

(1893, 53)
(120, 53)


(119, 53)

In [265]:
def interpolation(df):
  cols_clima = ['temp_max_dia', 'temp_min_dia', 'umidade_dia', 'chuva_dia']
  df[cols_clima] = df[cols_clima].interpolate(method='linear', limit_direction='both')
  return df

In [266]:
df_train = interpolation(df_train)
df_val = interpolation(df_val)
df_test = interpolation(df_test)

In [267]:
print(df_train.shape)
print(df_val.shape)
df_test.shape

(1893, 53)
(120, 53)


(119, 53)

In [268]:
df_train.columns

Index(['total_casos', 'score_gravidade', 'idade', 'chuva_dia', 'temp_max_dia',
       'temp_min_dia', 'umidade_dia', 'year', 'month', 'day_week',
       'chuva_acum_7d', 'faixa_chuva', 'temp_max_lag_7d', 'temp_min_lag_7d',
       'gravidade_lag_7d', 'casos_mm7d', 'chuva_acum_15d', 'temp_max_lag_15d',
       'temp_min_lag_15d', 'gravidade_lag_15d', 'casos_mm15d',
       'chuva_acum_21d', 'temp_max_lag_21d', 'temp_min_lag_21d',
       'gravidade_lag_21d', 'casos_mm21d', 'chuva_acum_30d',
       'temp_max_lag_30d', 'temp_min_lag_30d', 'gravidade_lag_30d',
       'casos_mm30d', 'chuva_acum_45d', 'temp_max_lag_45d', 'temp_min_lag_45d',
       'gravidade_lag_45d', 'casos_mm45d', 'chuva_acum_50d',
       'temp_max_lag_50d', 'temp_min_lag_50d', 'gravidade_lag_50d',
       'casos_mm50d', 'chuva_acum_60d', 'temp_max_lag_60d', 'temp_min_lag_60d',
       'gravidade_lag_60d', 'casos_mm60d', 'day_month', 'day_month_start',
       'day_month_end', 'day_quarter_start', 'day_quarter_end',
       'day_y

In [269]:
print_nan(df_train)
print_nan(df_val)
print_nan(df_test)

Colunas com valores NaN:
temp_min_lag_60d     60
gravidade_lag_60d    60
temp_max_lag_60d     60
faixa_chuva          59
casos_mm60d          59
chuva_acum_60d       59
temp_max_lag_50d     50
gravidade_lag_50d    50
temp_min_lag_50d     50
casos_mm50d          49
chuva_acum_50d       49
temp_min_lag_45d     45
gravidade_lag_45d    45
temp_max_lag_45d     45
casos_mm45d          44
chuva_acum_45d       44
temp_max_lag_30d     30
temp_min_lag_30d     30
gravidade_lag_30d    30
chuva_acum_30d       29
casos_mm30d          29
temp_max_lag_21d     21
gravidade_lag_21d    21
temp_min_lag_21d     21
casos_mm21d          20
chuva_acum_21d       20
gravidade_lag_15d    15
temp_min_lag_15d     15
temp_max_lag_15d     15
chuva_acum_15d       14
casos_mm15d          14
temp_max_lag_7d       7
gravidade_lag_7d      7
temp_min_lag_7d       7
chuva_acum_7d         6
casos_mm7d            6
dtype: int64
Colunas com valores NaN:
temp_min_lag_60d     60
gravidade_lag_60d    60
temp_max_lag_60d     60
f

In [270]:
# Criando faixas de umidade
def faixa_umidade(df):
  df['faixa_umidade'] = pd.cut(df['umidade_dia'], bins=[0, 40, 60, 80, 100],
                                      labels=['Muito Seco', 'Seco', 'Ideal', 'Úmido'])
  return df

In [271]:
df_train = faixa_umidade(df_train)
df_val = faixa_umidade(df_val)
df_test = faixa_umidade(df_test)

In [272]:
n_nans # quantidade máxima de NaNs esperado no val/test lags[-1]

60

In [273]:
df_train = df_train.iloc[n_nans:]
df_val = df_val.iloc[n_nans:]
df_test = df_test.iloc[n_nans:]

In [274]:
print_nan(df_train)
print_nan(df_val)
print_nan(df_test)

Nenhum valor NaN encontrado!
Nenhum valor NaN encontrado!
Nenhum valor NaN encontrado!


In [275]:
print(df_train.shape)
print(df_val.shape)
df_test.shape

(1833, 54)
(60, 54)


(59, 54)

In [276]:
def fourier_features(df, columns, periods, n_terms_list):
    t = np.arange(len(df))

    for col in columns:
        for period, n_terms in zip(periods, n_terms_list):
            period = int(period)  # ← garante que não vira array/lista

            for k in range(1, n_terms + 1):
                df[f"{col}_fourier_cos_p{period}_k{k}"] = np.cos(2 * np.pi * k * t / period)
                df[f"{col}_fourier_sin_p{period}_k{k}"] = np.sin(2 * np.pi * k * t / period)

    return df


In [277]:
seasonal_periods = [7, 30.44, 365.25]  # Semanal, Mensal e Anual
n_terms_list = [3, 1, 2] # Complexidade de cada onda

In [278]:
df_train = fourier_features(df_train, ['ciclos'], seasonal_periods, n_terms_list)
df_val = fourier_features(df_val, ['ciclos'], seasonal_periods, n_terms_list)
df_test = fourier_features(df_test, ['ciclos'], seasonal_periods, n_terms_list)

In [279]:
print_nan(df_train)
print_nan(df_val)
print_nan(df_test)

Nenhum valor NaN encontrado!
Nenhum valor NaN encontrado!
Nenhum valor NaN encontrado!


In [280]:
print(df_train.shape)
print(df_val.shape)
df_test.shape

(1833, 66)
(60, 66)


(59, 66)

In [286]:
df_train.columns

Index(['total_casos', 'score_gravidade', 'idade', 'chuva_dia', 'temp_max_dia',
       'temp_min_dia', 'umidade_dia', 'year', 'month', 'day_week',
       'chuva_acum_7d', 'faixa_chuva', 'temp_max_lag_7d', 'temp_min_lag_7d',
       'gravidade_lag_7d', 'casos_mm7d', 'chuva_acum_15d', 'temp_max_lag_15d',
       'temp_min_lag_15d', 'gravidade_lag_15d', 'casos_mm15d',
       'chuva_acum_21d', 'temp_max_lag_21d', 'temp_min_lag_21d',
       'gravidade_lag_21d', 'casos_mm21d', 'chuva_acum_30d',
       'temp_max_lag_30d', 'temp_min_lag_30d', 'gravidade_lag_30d',
       'casos_mm30d', 'chuva_acum_45d', 'temp_max_lag_45d', 'temp_min_lag_45d',
       'gravidade_lag_45d', 'casos_mm45d', 'chuva_acum_50d',
       'temp_max_lag_50d', 'temp_min_lag_50d', 'gravidade_lag_50d',
       'casos_mm50d', 'chuva_acum_60d', 'temp_max_lag_60d', 'temp_min_lag_60d',
       'gravidade_lag_60d', 'casos_mm60d', 'day_month', 'day_month_start',
       'day_month_end', 'day_quarter_start', 'day_quarter_end',
       'day_y

### Split

In [287]:
def split_target(df: pd.DataFrame, columns_target: str):
    y = df[columns_target]
    X = df.drop(columns=[columns_target])

    return X, y

In [288]:
# column_target = 'total_casos'

In [289]:
X_train, y_train = split_target(df_train, column_target)
X_val, y_val = split_target(df_val, column_target)
X_test, y_test = split_target(df_val, column_target)

In [290]:
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
X_test.shape, y_test.shape

(1833, 65) (1833,)
(60, 65) (60,)


((60, 65), (60,))

In [291]:
y_train.isna().sum().sum(), y_val.isna().sum().sum(), y_val.isna().sum().sum()

(np.int64(0), np.int64(0), np.int64(0))

# Scaler

In [294]:
X_train.columns

Index(['score_gravidade', 'idade', 'chuva_dia', 'temp_max_dia', 'temp_min_dia',
       'umidade_dia', 'year', 'month', 'day_week', 'chuva_acum_7d',
       'faixa_chuva', 'temp_max_lag_7d', 'temp_min_lag_7d', 'gravidade_lag_7d',
       'casos_mm7d', 'chuva_acum_15d', 'temp_max_lag_15d', 'temp_min_lag_15d',
       'gravidade_lag_15d', 'casos_mm15d', 'chuva_acum_21d',
       'temp_max_lag_21d', 'temp_min_lag_21d', 'gravidade_lag_21d',
       'casos_mm21d', 'chuva_acum_30d', 'temp_max_lag_30d', 'temp_min_lag_30d',
       'gravidade_lag_30d', 'casos_mm30d', 'chuva_acum_45d',
       'temp_max_lag_45d', 'temp_min_lag_45d', 'gravidade_lag_45d',
       'casos_mm45d', 'chuva_acum_50d', 'temp_max_lag_50d', 'temp_min_lag_50d',
       'gravidade_lag_50d', 'casos_mm50d', 'chuva_acum_60d',
       'temp_max_lag_60d', 'temp_min_lag_60d', 'gravidade_lag_60d',
       'casos_mm60d', 'day_month', 'day_month_start', 'day_month_end',
       'day_quarter_start', 'day_quarter_end', 'day_year_start',
       'da

In [298]:
fourier = X_train.filter(like='ciclo')
cols_fourier = fourier.columns.to_list()
cols_fourier

['ciclos_fourier_cos_p7_k1',
 'ciclos_fourier_sin_p7_k1',
 'ciclos_fourier_cos_p7_k2',
 'ciclos_fourier_sin_p7_k2',
 'ciclos_fourier_cos_p7_k3',
 'ciclos_fourier_sin_p7_k3',
 'ciclos_fourier_cos_p30_k1',
 'ciclos_fourier_sin_p30_k1',
 'ciclos_fourier_cos_p365_k1',
 'ciclos_fourier_sin_p365_k1',
 'ciclos_fourier_cos_p365_k2',
 'ciclos_fourier_sin_p365_k2']

In [314]:
X_train_fourier = X_train[cols_fourier]
X_val_fourier = X_val[cols_fourier]
X_test_fourier = X_test[cols_fourier]
X_test_fourier

,ciclos_fourier_cos_p7_k1,ciclos_fourier_sin_p7_k1,ciclos_fourier_cos_p7_k2,ciclos_fourier_sin_p7_k2,ciclos_fourier_cos_p7_k3,ciclos_fourier_sin_p7_k3,ciclos_fourier_cos_p30_k1,ciclos_fourier_sin_p30_k1,ciclos_fourier_cos_p365_k1,ciclos_fourier_sin_p365_k1,ciclos_fourier_cos_p365_k2,ciclos_fourier_sin_p365_k2
data,,,,,,,,,,,,
2025-10-26,1.000000,0.000000e+00,1.000000,0.000000e+00,1.000000,0.000000e+00,1.000000,0.000000e+00,1.000000,0.000000,1.000000,0.000000
2025-10-27,0.623490,7.818315e-01,-0.222521,9.749279e-01,-0.900969,4.338837e-01,0.978148,2.079117e-01,0.999852,0.017213,0.999407,0.034422
2025-10-28,-0.222521,9.749279e-01,-0.900969,-4.338837e-01,0.623490,-7.818315e-01,0.913545,4.067366e-01,0.999407,0.034422,0.997630,0.068802
2025-10-29,-0.900969,4.338837e-01,0.623490,-7.818315e-01,-0.222521,9.749279e-01,0.809017,5.877853e-01,0.998667,0.051620,0.994671,0.103102
2025-10-30,-0.900969,-4.338837e-01,0.623490,7.818315e-01,-0.222521,-9.749279e-01,0.669131,7.431448e-01,0.997630,0.068802,0.990532,0.137279
2025-10-31,-0.222521,-9.749279e-01,-0.900969,4.338837e-01,0.623490,7.818315e-01,0.500000,8.660254e-01,0.996298,0.085965,0.985220,0.171293
2025-11-01,0.623490,-7.818315e-01,-0.222521,-9.749279e-01,-0.900969,-4.338837e-01,0.309017,9.510565e-01,0.994671,0.103102,0.978740,0.205104
2025-11-02,1.000000,-2.449294e-16,1.000000,-4.898587e-16,1.000000,-7.347881e-16,0.104528,9.945219e-01,0.992749,0.120208,0.971100,0.238673
2025-11-03,0.623490,7.818315e-01,-0.222521,9.749279e-01,-0.900969,4.338837e-01,-0.104528,9.945219e-01,0.990532,0.137279,0.962309,0.271958


In [315]:
X_train_s = X_train.drop(columns=cols_fourier)
X_val_s = X_val.drop(columns=cols_fourier)
X_test_s = X_test.drop(columns=cols_fourier)

In [316]:
X_train_s.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1833 entries, 2020-03-01 to 2025-08-26
Data columns (total 53 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   score_gravidade    1833 non-null   float64 
 1   idade              1833 non-null   float64 
 2   chuva_dia          1833 non-null   float64 
 3   temp_max_dia       1833 non-null   float64 
 4   temp_min_dia       1833 non-null   float64 
 5   umidade_dia        1833 non-null   float64 
 6   year               1833 non-null   int32   
 7   month              1833 non-null   int32   
 8   day_week           1833 non-null   int32   
 9   chuva_acum_7d      1833 non-null   float64 
 10  faixa_chuva        1833 non-null   category
 11  temp_max_lag_7d    1833 non-null   float64 
 12  temp_min_lag_7d    1833 non-null   float64 
 13  gravidade_lag_7d   1833 non-null   float64 
 14  casos_mm7d         1833 non-null   float64 
 15  chuva_acum_15d     1833 non-null   fl

In [317]:
def encode_categorical_columns(df):
    df_encoded = df.copy()
    cat_cols = df_encoded.select_dtypes(include=['category', 'object']).columns

    for col in cat_cols:
        print(f"Transformando a coluna: {col}")
        df_encoded[col] = df_encoded[col].cat.codes

    return df_encoded

In [318]:
X_train_s = encode_categorical_columns(X_train_s)
X_val_s = encode_categorical_columns(X_val_s)
X_test_s = encode_categorical_columns(X_test_s)

Transformando a coluna: faixa_chuva
Transformando a coluna: faixa_umidade
Transformando a coluna: faixa_chuva
Transformando a coluna: faixa_umidade
Transformando a coluna: faixa_chuva
Transformando a coluna: faixa_umidade


In [320]:
scaler = StandardScaler()

In [321]:
X_train_scaled = scaler.fit_transform(X_train_s)
X_val_scaled = scaler.transform(X_val_s)
X_test_scaled = scaler.transform(X_test_s)

In [331]:
X_train_scaled_df = pd.DataFrame(X_train_scaled,
                                 columns=X_train_s.columns,
                                 index=X_train_s.index)

X_val_scaled_df = pd.DataFrame(X_val_scaled,
                                 columns=X_val_s.columns,
                                 index=X_val_s.index)

X_test_scaled_df = pd.DataFrame(X_test_scaled,
                                 columns=X_test_s.columns,
                                 index=X_test_s.index)

X_train_scaled_df

,score_gravidade,idade,chuva_dia,temp_max_dia,temp_min_dia,umidade_dia,year,month,day_week,chuva_acum_7d,...,gravidade_lag_60d,casos_mm60d,day_month,day_month_start,day_month_end,day_quarter_start,day_quarter_end,day_year_start,day_year_end,faixa_umidade
data,,,,,,,,,,,,,,,,,,,,,
2020-03-01,0.869640,-0.446933,14.949865,-1.420727,0.896071,1.918765,-1.592496,-1.030817,1.496151,4.871888,...,0.711506,-0.367691,-1.674526,5.344578,-0.182368,-0.107654,-0.099586,-0.046765,-0.03305,1.135899
2020-03-02,0.139131,0.094649,1.376479,-1.444362,0.348351,1.715908,-1.592496,-1.030817,-1.482069,5.399808,...,-0.932760,-0.368604,-1.560000,-0.187106,-0.182368,-0.107654,-0.099586,-0.046765,-0.03305,1.135899
2020-03-03,0.827495,0.081831,0.030893,-0.546237,0.445008,0.929490,-1.592496,-1.030817,-0.985699,5.467631,...,0.198373,-0.369516,-1.445474,-0.187106,-0.182368,-0.107654,-0.099586,-0.046765,-0.03305,1.135899
2020-03-04,-0.116547,-1.146823,-0.257028,-0.688046,0.799415,0.616868,-1.592496,-1.030817,-0.489329,5.462132,...,1.407901,-0.369516,-1.330949,-0.187106,-0.182368,-0.107654,-0.099586,-0.046765,-0.03305,1.135899
2020-03-05,0.371566,0.155246,-0.227648,-0.309888,-0.296024,0.183365,-1.592496,-1.030817,0.007041,5.005701,...,0.556752,-0.369212,-1.216423,-0.187106,-0.182368,-0.107654,-0.099586,-0.046765,-0.03305,-0.753487
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08-22,0.991391,-0.743182,-0.345166,0.588237,-0.811525,-0.152878,1.570067,0.491245,0.503411,-0.733594,...,-0.003215,-0.339548,0.730516,-0.187106,-0.182368,-0.107654,-0.099586,-0.046765,-0.03305,-0.753487
2025-08-23,-1.139259,0.563325,-0.345166,1.108204,-1.069276,-2.801135,1.570067,0.491245,0.999781,-0.755591,...,-0.607979,-0.340612,0.845042,-0.187106,-0.182368,-0.107654,-0.099586,-0.046765,-0.03305,-2.642872
2025-08-24,-0.286999,-0.585800,-0.345166,-0.309888,-0.618212,-0.539140,1.570067,0.491245,1.496151,-0.755591,...,-0.507185,-0.341373,0.959568,-0.187106,-0.182368,-0.107654,-0.099586,-0.046765,-0.03305,-0.753487


In [332]:
X_train_scaled = pd.concat([X_train_scaled_df, X_train_fourier], axis=1)
X_val_scaled = pd.concat([X_val_scaled_df, X_val_fourier], axis=1)
X_test_scaled = pd.concat([X_test_scaled_df, X_test_fourier], axis=1)

In [333]:
X_train_scaled.shape, X_train.shape

((1833, 65), (1833, 65))

In [334]:
X_val_scaled.shape, X_val.shape

((60, 65), (60, 65))

In [335]:
X_test_scaled.shape, X_test.shape

((60, 65), (60, 65))

In [337]:
X_test_scaled.head()

,score_gravidade,idade,chuva_dia,temp_max_dia,temp_min_dia,umidade_dia,year,month,day_week,chuva_acum_7d,...,ciclos_fourier_cos_p7_k2,ciclos_fourier_sin_p7_k2,ciclos_fourier_cos_p7_k3,ciclos_fourier_sin_p7_k3,ciclos_fourier_cos_p30_k1,ciclos_fourier_sin_p30_k1,ciclos_fourier_cos_p365_k1,ciclos_fourier_sin_p365_k1,ciclos_fourier_cos_p365_k2,ciclos_fourier_sin_p365_k2
data,,,,,,,,,,,,,,,,,,,,,
2025-10-26,-0.020668,0.037366,-0.345166,1.628171,-0.231587,-1.986928,1.570067,1.100069,1.496151,-0.652940,...,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
2025-10-27,0.139131,-0.699671,-0.274655,0.162809,-0.360462,0.734969,1.570067,1.100069,-1.482069,-0.733594,...,-0.222521,0.974928,-0.900969,0.433884,0.978148,0.207912,0.999852,0.017213,0.999407,0.034422
2025-10-28,-0.500064,0.240459,0.007389,-0.238984,0.316133,1.475536,1.570067,1.100069,-0.985699,-0.623611,...,-0.900969,-0.433884,0.623490,-0.781831,0.913545,0.406737,0.999407,0.034422,0.997630,0.068802
2025-10-29,-0.500064,-0.462555,-0.339291,0.493697,0.122820,0.689813,1.570067,1.100069,-0.489329,-0.621778,...,0.623490,-0.781831,-0.222521,0.974928,0.809017,0.587785,0.998667,0.051620,0.994671,0.103102
2025-10-30,0.139131,-0.419159,0.365820,-1.184378,0.090601,1.037865,1.570067,1.100069,0.007041,-0.399978,...,0.623490,0.781831,-0.222521,-0.974928,0.669131,0.743145,0.997630,0.068802,0.990532,0.137279


In [338]:
print_nan(X_train_scaled)
print_nan(X_val_scaled)
print_nan(X_test_scaled)

Nenhum valor NaN encontrado!
Nenhum valor NaN encontrado!
Nenhum valor NaN encontrado!


# Feature Selection

In [ ]:
def feature_selection(X_train, y_train, X_test, model, scoring='neg_root_mean_squared_error', cv=3, threshold=0.001):
    """
    Executa feature selection usando RecursiveFeatureElimination com o modelo cru.
    """
    estimator = clone(model)

    feat_selector = RecursiveFeatureElimination(
        variables=None,
        estimator=estimator,
        scoring=scoring,
        threshold=threshold,
        cv=cv,
    )

    feat_selector.fit(X_train.copy(), y_train)
    X_train_fs = feat_selector.transform(X_train.copy())
    X_test_fs = feat_selector.transform(X_test.copy())

    return X_train_fs, X_test_fs, feat_selector

# Hyperparam Optimization

In [ ]:
def optimize_model(X_train, y_train, model_name='XGBoost', n_trials=50, n_splits=10, random_state=42):
    """
    Realiza hyperparameter optimization usando Optuna para modelos clássicos de regressão.
    Retorna o melhor dicionário de parâmetros (best_params).
    """

    tscv = TimeSeriesSplit(n_splits=n_splits)

    # Função objetivo para Optuna
    def objective(trial):
        if model_name == 'XGBoost':
            parans = {
                "device": "cuda",
                "tree_method": "hist",
                "n_estimators": trial.suggest_int('n_estimators', 100, 500),
                "max_depth": trial.suggest_int('max_depth', 3, 8),
                "learning_rate": trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
                "subsample": trial.suggest_float('subsample', 0.6, 0.9),
                "colsample_bytree": trial.suggest_float('colsample_bytree', 0.5, 1.0),
                "n_jobs": -1,
                "random_state": random_state
            }
            n_estimators = trial.suggest_int('n_estimators', 100, 500)
            max_depth = trial.suggest_int('max_depth', 3, 8)
            learning_rate = trial.suggest_float('learning_rate', 0.01, 0.1, log=True)
            subsample = trial.suggest_float('subsample', 0.6, 0.9)
            colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)
            model = XGBRegressor(**params)

        elif model_name == 'LightGBM':
            parans = {
                "device": "gpu",
                "gpu_platform_id": 0,
                "gpu_device_id": 0,
                "n_estimators": trial.suggest_int('n_estimators', 100, 500),
                "max_depth": trial.suggest_int('max_depth', 3, 8),
                "learning_rate": trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
                "num_leaves": trial.suggest_int('num_leaves', 20, 60),
                "subsample": trial.suggest_float('subsample', 0.5, 1.0),
                "colsample_bytree": trial.suggest_float('colsample_bytree', 0.5, 1.0),
                "n_jobs": -1,
                "random_state": random_state
            }
            model = LGBMRegressor(**params)
        else:
            raise ValueError(f'Model {model_name} não suportado.')

        # Avaliação com TimeSeriesSplit
        scores = cross_val_score(
            model, X_train, y_train,
            cv=tscv,
            scoring=make_scorer(lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred))),
            n_jobs=-1
        )

        return np.mean(scores)  # Optuna minimiza RMSE médio

    # Criar estudo e otimizar
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    # print(f"Melhores parâmetros para {model_name}: {study.best_params}")
    # print(f"Melhor RMSE médio (CV): {study.best_value:.4f}")

    # Retorna os melhores parâmetros (para Linear será vazio)
    return study.best_params

# Forecasting

In [339]:
def forecasting(X_train, y_train, X_val, model_name, n_trials=10, n_splits=15):
  """
  Executa o fluxo completo:
  - Otimiza hiperparâmetros com Optuna
  - Seleciona features com modelo cru
  - Treina e prevê com modelo otimizado
  """
  X_train_ = X_train.copy()
  X_val_ = X_val.copy()
  y_train_ = y_train.copy()

  # ----  Hyperparameter tuning ----
  best_params = optimize_model(X_train_, y_train_, model_name=model_name, n_trials=n_trials, n_splits=n_splits)
  if best_params is None:
      best_params = {}

  # ---- Criação do modelo otimizado ----
  model_map = {
      'XGBoost': XGBRegressor,
      'LightGBM': LGBMRegressor
  }

  if model_name not in model_map:
      raise ValueError(f"Modelo {model_name} não suportado.")

  # modelo otimizado (para previsão final)
  model_opt = model_map[model_name](**best_params)

  # modelo cru (para seleção de features)
  model_raw = model_map[model_name]()

  # ---- Feature selection com modelo cru ----
  X_train_fs, X_val_fs, feat_selected = feature_selection(
      X_train_, y_train_, X_val_, model_raw, scoring='neg_root_mean_squared_error', cv=5, threshold=0.001
  )

  # ---- Treinamento e previsão ----
  model_opt.fit(X_train_fs, y_train_)
  y_pred = model_opt.predict(X_val_fs)

  return y_pred, best_params, feat_selected, model_opt

# Metrics/Results

In [340]:
df_result = pd.DataFrame()
forecast_metric = dict()

In [341]:
def get_erros_metric(x_test, pedrict, forecast_metric, time):
  mse = mean_squared_error(x_test,pedrict)
  forecast_metric.update({'MSE':mse})

  RMSE = np.sqrt(mse)
  forecast_metric.update({'RMSE':RMSE})

  r2 = r2_score(y_test, y_pred_final)
  forecast_metric.update({'R2':r2})

  forecast_metric.update({'Seconds':round(time,6)})

  return forecast_metric

In [342]:
def add_resultado(df_result: pd.DataFrame, forecast_method: str, method: dict) -> pd.DataFrame:
    # Se ainda não tem coluna 'Type', cria
    if "Type" not in df_result.columns:
        df_result["Type"] = None

    # Verifica se já existe linha para esse método
    mask = df_result["Type"] == forecast_method

    if mask.any():
        # Atualiza as métricas na linha existente
        for key, value in method.items():
            df_result.loc[mask, key] = value
    else:
        # Cria nova linha com Type e métricas
        new_row = {"Type": forecast_method}
        new_row.update(method)
        df_result = pd.concat([df_result, pd.DataFrame([new_row])], ignore_index=True)

    forecast_metric.clear()
    return df_result

# XGBoost

In [343]:
start_time = time.time()
model_name = 'XGBoost'
y_pred_xg_boost, best_params_xg_boost, feat_selected_xg_boost, model_xg_boost = forecasting(X_train_scaled, y_train, X_val_scaled, model_name, 50)
end_time = time.time()

[I 2026-05-10 22:59:33,222] A new study created in memory with name: no-name-2ec33871-04af-49a5-8d58-09c107b227bf
[I 2026-05-10 23:00:35,804] Trial 0 finished with value: 44.84136189261969 and parameters: {'n_estimators': 275, 'max_depth': 7, 'learning_rate': 0.032456534346239116, 'subsample': 0.8230295717519495, 'colsample_bytree': 0.7987770605035214}. Best is trial 0 with value: 44.84136189261969.
[I 2026-05-10 23:00:49,750] Trial 1 finished with value: 44.239392957079716 and parameters: {'n_estimators': 278, 'max_depth': 5, 'learning_rate': 0.036493651445566266, 'subsample': 0.8105828176069343, 'colsample_bytree': 0.5049652023317979}. Best is trial 1 with value: 44.239392957079716.
[I 2026-05-10 23:01:11,956] Trial 2 finished with value: 44.20352399265164 and parameters: {'n_estimators': 380, 'max_depth': 5, 'learning_rate': 0.010273345875376101, 'subsample': 0.6582304468841544, 'colsample_bytree': 0.7340266669471758}. Best is trial 2 with value: 44.20352399265164.
[I 2026-05-10 23:

AttributeError: 'super' object has no attribute '__sklearn_tags__'

In [344]:
print(f'{model_name}: {best_params_xg_boost}')

NameError: name 'best_params_xg_boost' is not defined

In [345]:
feat_selected_xg_boost.initial_model_performance_

NameError: name 'feat_selected_xg_boost' is not defined

In [ ]:
len(feat_selected_xg_boost.features_to_drop_)

In [ ]:
feat_selected_xg_boost.feature_importances_.plot.bar(figsize=(20,6))
plt.xlabel('Features')
plt.ylabel('Importance')
plt.title(label=f'{model_name}')
plt.show()

In [ ]:
pd.Series(feat_selected_xg_boost.performance_drifts_).plot.bar(figsize=(20,6))
plt.xlabel('Features')
plt.ylabel('Performance change when feature was added')
plt.title(label=f'{model_name}')
plt.show()

In [ ]:
# metrics = get_erros_metric(y_val, y_pred_xg_boost, forecast_metric, (end_time - start_time))
# df_result = add_resultado(df_result, model_name, metrics)
# df_result

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(y_val.index, y_val, label='Real',  color='green', lw=2)
plt.plot(y_val.index, y_pred_xg_boost, label='Previsto', color='orange', lw=2)
plt.title(f"{model_name} - Previsão vs Real (Validação)")
plt.xlabel("Date")
plt.ylabel("Close")
plt.legend()
plt.show()

In [ ]:
y_pred_final = model_xg_boost.predict(X_test_scaled)

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(y_test.index, y_test, label='Real',  color='green', lw=2)
plt.plot(y_test.index, y_pred_final, label='Previsto', color='orange', lw=2)
plt.title(f"{model_name} - Previsão vs Real (Test)")
plt.xlabel("Date")
plt.ylabel("Close")
plt.legend()
plt.show()

In [ ]:
# Em séries temporais, como o seu projeto de dengue/saúde, após você validar tudo
# e chegar à conclusão de que o modelo é bom usando o `X_val`, existe um passo extra muito comum:

# **Retreinar o modelo com Treino + Validação:**
# Antes de fazer a predição final no Teste, muitos cientistas de dados juntam o Treino e
# a Validação em um único bloco e treinam o modelo uma última vez com esses dados combinados.
# Porque assim o modelo terá visto os dados mais recentes (da validação)
# antes de tentar prever o teste, o que é valioso para capturar as tendências mais próximas de 2026.

# Opcional: Juntar treino e val para o último treino antes do teste real
X_final_train = pd.concat([X_train_scaled, X_val_scaled])
y_final_train = pd.concat([y_train, y_val])

# modelo.fit(X_final_train, y_final_train)
y_pred_final_2 = model_xg_boost.predict(X_test_scaled) # Comparar com y_test

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(y_test.index, y_test, label='Real',  color='green', lw=2)
plt.plot(y_test.index, y_pred_final_2, label='Previsto', color='orange', lw=2)
plt.title(f"{model_name} - Previsão vs Real (Test 2)")
plt.xlabel("Date")
plt.ylabel("Close")
plt.legend()
plt.show()